In [1]:
import os
import json
import random
import numpy as np
import xml.etree.ElementTree as ET
from sumolib import net

SUMO_HOME = "C:\\Program Files (x86)\\Eclipse\\Sumo"

NET_FILE = "map.net.xml"
TAZ_FILE = "grids.taz.xml"

OD_XML = "od.xml"
TRIPS_FILE = "trips.xml"
ROUTES_FILE = "routes.rou.xml"
EDGES_FILE = "edges.xml"

In [2]:
# !python "C:\Program Files (x86)\Eclipse\Sumo\tools\district\gridDistricts.py" -n map.net.xml -o grids.taz.xml -w 500
# !sumo-gui -c sim.sumocfg

In [3]:
net_obj = net.readNet(NET_FILE)

tree = ET.parse(TAZ_FILE)
root = tree.getroot()

zones = [t.attrib["id"] for t in root.findall("taz")]
n = len(zones)

zone_index = {z: i for i, z in enumerate(zones)}

print("Zones:", n)

Zones: 12


In [4]:
neshan_weights = json.load(open("data/neshan-sample-weights.json"))

print("Target edges:", len(neshan_weights))

Target edges: 360


In [5]:
def save_od_to_xml(od_matrix, filename=OD_XML):
    root = ET.Element("data")
    interval = ET.SubElement(root, "interval", {"begin": "0", "end": "3600"})

    for i, zi in enumerate(zones):
        for j, zj in enumerate(zones):
            if i == j:
                continue

            count = int(od_matrix[i][j])
            if count <= 0:
                continue

            ET.SubElement(interval, "tazRelation", {
                "from": zi,
                "to": zj,
                "count": str(count)
            })

    ET.ElementTree(root).write(filename, encoding="utf-8", xml_declaration=True)

In [6]:
def run_sumo_pipeline():
    # Step 1: OD → trips
    os.system(f"od2trips -z {OD_XML} -n {TAZ_FILE} -o {TRIPS_FILE}")

    # Step 2: trips → routes
    os.system(f"duarouter -n {NET_FILE} -r {TRIPS_FILE} -o {ROUTES_FILE} --ignore-errors true")

    # Step 3: run simulation (assuming sim.sumocfg exists)
    os.system(f"sumo -c sim.sumocfg --summary-output {EDGES_FILE}")

In [7]:
def load_edge_relative_speed(file_path=EDGES_FILE):
    tree = ET.parse(file_path)
    root = tree.getroot()

    edge_tt = {}

    for interval in root.findall("interval"):
        for edge in interval.findall("edge"):
            eid = edge.get("id")
            tt = edge.get("speedRelative")

            if eid and tt:
                edge_tt[eid] = float(tt)

    return edge_tt

In [8]:
def fitness(sim, target):
    common = set(sim.keys()) & set(target.keys())

    if not common:
        return 1e9

    error = 0.0
    for e in common:
        error += (sim[e] - target[e]) ** 2

    return error / len(common)

In [ ]:
def random_od():
    m = np.zeros((n, n), dtype=int)

    for i in range(n):
        for j in range(n):
            if i != j:
                m[i][j] = random.randint(0, 100)

    return m

In [10]:
n

12

In [11]:
def crossover(a, b):
    child = np.zeros_like(a)

    for i in range(n):
        for j in range(n):
            child[i][j] = a[i][j] if random.random() < 0.5 else b[i][j]

    return child

In [12]:
def mutate(m, rate=0.05):
    for i in range(n):
        for j in range(n):
            if i != j and random.random() < rate:
                m[i][j] = random.randint(0, 100)
    return m

In [13]:
def evaluate_od(od):
    save_od_to_xml(od)
    run_sumo_pipeline()
    sim = load_edge_relative_speed()
    return fitness(sim, neshan_weights)

In [14]:
POP_SIZE = 6
GENERATIONS = 7

population = [random_od() for _ in range(POP_SIZE)]

best_solution = None
best_score = float("inf")

for gen in range(GENERATIONS):
    print(f"\n=== Generation {gen} ===")

    scores = []

    for i, ind in enumerate(population):
        print(f"Evaluating individual {i}")

        score = evaluate_od(ind)
        scores.append(score)

        print("Score:", score)

        if score < best_score:
            best_score = score
            best_solution = ind.copy()

    print("Best in gen:", min(scores))

    # Selection (top 50%)
    idx = np.argsort(scores)
    selected = [population[i] for i in idx[:POP_SIZE // 2]]

    # Create next generation
    new_pop = []

    while len(new_pop) < POP_SIZE:
        p1, p2 = random.sample(selected, 2)

        child = crossover(p1, p2)
        child = mutate(child)

        new_pop.append(child)

    population = new_pop

print("\nDONE")
print("Best score:", best_score)


=== Generation 0 ===
Evaluating individual 0
Score: 0.05675418994413401
Evaluating individual 1
Score: 0.0357871508379888
Evaluating individual 2
Score: 0.06489777158774371
Evaluating individual 3
Score: 0.056063788300835624
Evaluating individual 4
Score: 0.04165167597765361
Evaluating individual 5
Score: 0.03725742296918766
Best in gen: 0.0357871508379888

=== Generation 1 ===
Evaluating individual 0
Score: 0.1291387186629528
Evaluating individual 1
Score: 0.04342089136490247
Evaluating individual 2
Score: 0.06846685236768794
Evaluating individual 3
Score: 0.043892997198879555
Evaluating individual 4
Score: 0.055237325905292456
Evaluating individual 5
Score: 0.034411516853932574
Best in gen: 0.034411516853932574

=== Generation 2 ===
Evaluating individual 0
Score: 0.04977715877437326
Evaluating individual 1
Score: 0.10941820728291331
Evaluating individual 2
Score: 0.05931166666666666
Evaluating individual 3
Score: 0.14305611111111136
Evaluating individual 4
Score: 0.03693649025069637

In [15]:
best_solution

array([[  0,  61,  64,  38,  20,   7,  43,  85,  64,  82,  27,  13],
       [ 95,   0,  68,  89,  10,  87,  76,  55,  88,  74,  95,  39],
       [ 25,  30,   0,  95,  66,   3,  95,  34,  63,  45,  71,  47],
       [  5,  68,  87,   0,  69,  51,  32,  29,  10,  94,  40,  20],
       [ 43,  78, 100,  68,   0,  22,  90,  60,  90,  22,  81,  66],
       [ 37,  46,  52,  67,  81,   0,  37,  69,  14,  51,  66,  50],
       [ 10,   9,  98,  44,  48,   1,   0,  66,   3,  17,   9,  14],
       [ 27,  24,  68,  65,  56,  44,  33,   0,  86,  20,  68,  33],
       [ 75,  15,  76,  42,  31,  51,  70,  57,   0,  26,  85,  97],
       [ 64,  76,  10,  26,   3,  97,  48,  26,  52,   0,  47,  96],
       [ 54,  58,  20,  65,   8,   5,  32,  69,  54,  11,   0,  69],
       [ 70,   0,  89,  96,  12,  90,  76,  11,  94,  93,  54,   0]])

In [16]:
np.save("best_od.npy", best_solution)

In [17]:
def save_od_to_sumo_xml(od_matrix, zones, output_file="final_od.xml"):
    root = ET.Element("data")

    interval = ET.SubElement(root, "interval", {
        "begin": "0",
        "end": "3600"
    })

    for i, from_zone in enumerate(zones):
        for j, to_zone in enumerate(zones):

            if i == j:
                continue

            count = int(od_matrix[i][j])

            if count <= 0:
                continue

            ET.SubElement(interval, "tazRelation", {
                "from": str(from_zone),
                "to": str(to_zone),
                "count": str(count)
            })

    tree = ET.ElementTree(root)
    tree.write(output_file, encoding="utf-8", xml_declaration=True)

    print(f"Saved OD to {output_file}")


save_od_to_sumo_xml(best_solution, zones, "final_od.xml")

Saved OD to final_od.xml
